# DATASCI 451 - Practical Applications

## A. Missing Data Prediction (Leave-One-Out Cross-Validation)
## B. Decision Support Framework

---

In [ ]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries loaded!")

In [ ]:
# Load data
df = pd.read_csv('data/selected_stations_monthly.csv')
station_names = df['short_name'].unique()
n_stations = len(station_names)
n_months = 4

# Station coordinates for later use
coords = {
    'Ann Arbor UMich': (42.28, -83.74),
    'Atlanta MI': (45.00, -84.14),
    'Bad Axe': (43.80, -83.00),
    'Bergland Dam': (46.59, -89.57),
    'Traverse City': (44.74, -85.58),
    'Pontiac Airport': (42.67, -83.42),
    'Gwinn Sawyer AFB': (46.35, -87.40),
    'Iron Mountain': (45.82, -88.11),
}

print(f"Data: {len(df)} observations, {n_stations} stations, {n_months} months")
df.head()

---
# Part A: Missing Data Prediction

**Question**: If a weather station's data is missing for a particular month, how well can we predict it?

**Method**: Leave-One-Out Cross-Validation
1. Remove one observation (station-month pair)
2. Fit both No Pooling and Hierarchical models on remaining data
3. Predict the held-out observation
4. Compare prediction errors

**Hypothesis**: Hierarchical model should perform better because it can "borrow strength" from other stations.

In [ ]:
def fit_and_predict_hierarchical(df_train, station_to_predict, month_to_predict, 
                                  station_names, n_samples=1000):
    """
    Fit hierarchical model and predict for a held-out station-month.
    """
    # Prepare training data
    station_idx = {name: i for i, name in enumerate(station_names)}
    df_train = df_train.copy()
    df_train['station_id'] = df_train['short_name'].map(station_idx)
    df_train['month_id'] = df_train['Month'] - 1
    
    y = df_train['TAVG_mean'].values
    station = df_train['station_id'].values
    month = df_train['month_id'].values
    
    n_stations = len(station_names)
    n_months = 4
    
    with pm.Model() as model:
        mu_alpha = pm.Normal('mu_alpha', mu=25, sigma=20)
        tau = pm.HalfCauchy('tau', beta=10)
        alpha_offset = pm.Normal('alpha_offset', mu=0, sigma=1, shape=n_stations)
        alpha = pm.Deterministic('alpha', mu_alpha + tau * alpha_offset)
        beta = pm.Normal('beta', mu=0, sigma=15, shape=n_months)
        sigma = pm.HalfCauchy('sigma', beta=10)
        
        mu_y = alpha[station] + beta[month]
        y_obs = pm.Normal('y_obs', mu=mu_y, sigma=sigma, observed=y)
        
        trace = pm.sample(n_samples, tune=500, cores=2, random_seed=42,
                         return_inferencedata=True, progressbar=False)
    
    # Predict for held-out observation
    alpha_samples = trace.posterior['alpha'].values.reshape(-1, n_stations)
    beta_samples = trace.posterior['beta'].values.reshape(-1, n_months)
    sigma_samples = trace.posterior['sigma'].values.flatten()
    
    pred_station_idx = station_idx[station_to_predict]
    pred_month_idx = month_to_predict - 1
    
    # Posterior predictive
    mu_pred = alpha_samples[:, pred_station_idx] + beta_samples[:, pred_month_idx]
    y_pred = np.random.normal(mu_pred, sigma_samples)
    
    return y_pred.mean(), y_pred.std(), np.percentile(y_pred, [2.5, 97.5])

In [ ]:
def fit_and_predict_no_pooling(df_train, station_to_predict, month_to_predict,
                                station_names, n_samples=1000):
    """
    Fit no pooling model and predict for a held-out station-month.
    Note: If station has no data for that month, prediction uses prior only.
    """
    station_idx = {name: i for i, name in enumerate(station_names)}
    df_train = df_train.copy()
    df_train['station_id'] = df_train['short_name'].map(station_idx)
    df_train['month_id'] = df_train['Month'] - 1
    
    y = df_train['TAVG_mean'].values
    station = df_train['station_id'].values
    month = df_train['month_id'].values
    
    n_stations = len(station_names)
    n_months = 4
    
    with pm.Model() as model:
        alpha = pm.Normal('alpha', mu=25, sigma=20, shape=n_stations)
        beta = pm.Normal('beta', mu=0, sigma=15, shape=n_months)
        sigma = pm.HalfCauchy('sigma', beta=10)
        
        mu_y = alpha[station] + beta[month]
        y_obs = pm.Normal('y_obs', mu=mu_y, sigma=sigma, observed=y)
        
        trace = pm.sample(n_samples, tune=500, cores=2, random_seed=42,
                         return_inferencedata=True, progressbar=False)
    
    alpha_samples = trace.posterior['alpha'].values.reshape(-1, n_stations)
    beta_samples = trace.posterior['beta'].values.reshape(-1, n_months)
    sigma_samples = trace.posterior['sigma'].values.flatten()
    
    pred_station_idx = station_idx[station_to_predict]
    pred_month_idx = month_to_predict - 1
    
    mu_pred = alpha_samples[:, pred_station_idx] + beta_samples[:, pred_month_idx]
    y_pred = np.random.normal(mu_pred, sigma_samples)
    
    return y_pred.mean(), y_pred.std(), np.percentile(y_pred, [2.5, 97.5])

In [ ]:
# Run LOO-CV for a subset of observations (full LOO would take too long)
# Select strategic observations: one per station, mix of months

test_cases = [
    ('Bergland Dam', 1),      # Coldest station, coldest month
    ('Ann Arbor UMich', 4),   # Warmest station, warmest month
    ('Gwinn Sawyer AFB', 2),  # Middle station, Feb
    ('Bad Axe', 3),           # Middle station, Mar
    ('Iron Mountain', 4),     # UP station, Apr
    ('Traverse City', 1),     # Lake effect station, Jan
]

results = []

print("Running Leave-One-Out Cross-Validation...")
print("="*70)

for station, month in test_cases:
    print(f"\nHeld out: {station}, Month {month}")
    
    # True value
    true_val = df[(df['short_name'] == station) & (df['Month'] == month)]['TAVG_mean'].values[0]
    
    # Training data (remove held-out observation)
    df_train = df[~((df['short_name'] == station) & (df['Month'] == month))].copy()
    
    # Fit and predict with both models
    hier_mean, hier_std, hier_ci = fit_and_predict_hierarchical(
        df_train, station, month, station_names)
    
    np_mean, np_std, np_ci = fit_and_predict_no_pooling(
        df_train, station, month, station_names)
    
    # Calculate errors
    hier_error = abs(hier_mean - true_val)
    np_error = abs(np_mean - true_val)
    
    # Check if true value is in CI
    hier_in_ci = hier_ci[0] <= true_val <= hier_ci[1]
    np_in_ci = np_ci[0] <= true_val <= np_ci[1]
    
    results.append({
        'station': station,
        'month': month,
        'true': true_val,
        'hier_pred': hier_mean,
        'hier_error': hier_error,
        'hier_ci_width': hier_ci[1] - hier_ci[0],
        'hier_in_ci': hier_in_ci,
        'np_pred': np_mean,
        'np_error': np_error,
        'np_ci_width': np_ci[1] - np_ci[0],
        'np_in_ci': np_in_ci
    })
    
    print(f"  True value: {true_val:.1f}°F")
    print(f"  Hierarchical: {hier_mean:.1f}°F (error: {hier_error:.1f}°F)")
    print(f"  No Pooling:   {np_mean:.1f}°F (error: {np_error:.1f}°F)")

results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("LOO-CV Complete!")

In [ ]:
# Summary of LOO-CV results
print("\nLOO-CV RESULTS SUMMARY")
print("="*70)

print(f"\n{'Station':<20} {'Month':>6} {'True':>8} {'Hier':>8} {'NP':>8} {'Hier Err':>10} {'NP Err':>10}")
print("-"*70)
for _, row in results_df.iterrows():
    print(f"{row['station']:<20} {row['month']:>6} {row['true']:>8.1f} "
          f"{row['hier_pred']:>8.1f} {row['np_pred']:>8.1f} "
          f"{row['hier_error']:>10.2f} {row['np_error']:>10.2f}")

print("-"*70)
print(f"{'MEAN':<20} {'':<6} {'':<8} {'':<8} {'':<8} "
      f"{results_df['hier_error'].mean():>10.2f} {results_df['np_error'].mean():>10.2f}")

print(f"\nCoverage (true in 95% CI):")
print(f"  Hierarchical: {results_df['hier_in_ci'].sum()}/{len(results_df)} ({results_df['hier_in_ci'].mean()*100:.0f}%)")
print(f"  No Pooling:   {results_df['np_in_ci'].sum()}/{len(results_df)} ({results_df['np_in_ci'].mean()*100:.0f}%)")

In [ ]:
# Visualize LOO-CV results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Prediction errors comparison
ax1 = axes[0]
x = np.arange(len(results_df))
width = 0.35

bars1 = ax1.bar(x - width/2, results_df['hier_error'], width, label='Hierarchical', color='coral')
bars2 = ax1.bar(x + width/2, results_df['np_error'], width, label='No Pooling', color='steelblue')

ax1.set_xlabel('Test Case')
ax1.set_ylabel('Absolute Prediction Error (°F)')
ax1.set_title('LOO-CV: Prediction Errors by Model')
ax1.set_xticks(x)
labels = [f"{row['station'][:10]}\n({row['month']})" for _, row in results_df.iterrows()]
ax1.set_xticklabels(labels, fontsize=9)
ax1.legend()
ax1.axhline(results_df['hier_error'].mean(), color='coral', linestyle='--', alpha=0.7)
ax1.axhline(results_df['np_error'].mean(), color='steelblue', linestyle='--', alpha=0.7)

# Plot 2: Predicted vs True
ax2 = axes[1]
ax2.scatter(results_df['true'], results_df['hier_pred'], s=100, c='coral', 
            edgecolors='black', label='Hierarchical', zorder=5)
ax2.scatter(results_df['true'], results_df['np_pred'], s=100, c='steelblue',
            edgecolors='black', marker='s', label='No Pooling', zorder=5)

lims = [results_df['true'].min() - 5, results_df['true'].max() + 5]
ax2.plot(lims, lims, 'k--', alpha=0.5, label='Perfect prediction')
ax2.set_xlabel('True Temperature (°F)')
ax2.set_ylabel('Predicted Temperature (°F)')
ax2.set_title('LOO-CV: Predicted vs True')
ax2.legend()
ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_aspect('equal')

plt.tight_layout()
plt.savefig('plots/17_loocv_results.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: plots/17_loocv_results.png")

### Part A Conclusion

**Key Finding**: When data is missing, hierarchical models can still make reasonable predictions by borrowing information from:
1. Other months at the same station
2. The same month at other stations
3. The overall population mean

This is especially valuable for new or poorly monitored stations.

---
# Part B: Decision Support Framework

Transform model outputs into actionable decision support:
1. **Energy Planning**: Probability of freezing temperatures
2. **Agriculture**: Safe planting date probabilities
3. **Road Safety**: Expected icy days

In [ ]:
# Load full model trace
trace_hier = az.from_netcdf('data/trace_hierarchical.nc')

# Extract posterior samples
alpha_samples = trace_hier.posterior['alpha'].values.reshape(-1, n_stations)
beta_samples = trace_hier.posterior['beta'].values.reshape(-1, n_months)
sigma_samples = trace_hier.posterior['sigma'].values.flatten()

n_posterior = len(sigma_samples)
print(f"Posterior samples: {n_posterior}")

## B1. Energy Planning: Freezing Probability

**Decision**: Energy companies need to know the probability of freezing temperatures to plan heating demand.

In [ ]:
def calc_freezing_prob(alpha_samples, beta_samples, sigma_samples, 
                       station_idx, month_idx, threshold=32):
    """
    Calculate P(Temperature < threshold) for a station-month.
    """
    mu = alpha_samples[:, station_idx] + beta_samples[:, month_idx]
    # Probability that a random day is below threshold
    prob = stats.norm.cdf(threshold, loc=mu, scale=sigma_samples)
    return prob.mean(), np.percentile(prob, [2.5, 97.5])

# Calculate freezing probabilities for all station-months
freezing_probs = []

station_idx_map = {name: i for i, name in enumerate(station_names)}
month_names = ['January', 'February', 'March', 'April']

print("FREEZING PROBABILITY P(T < 32°F)")
print("="*70)
print(f"\n{'Station':<20} {'Jan':>10} {'Feb':>10} {'Mar':>10} {'Apr':>10}")
print("-"*70)

for station in station_names:
    row_probs = []
    for month_idx in range(4):
        prob, ci = calc_freezing_prob(alpha_samples, beta_samples, sigma_samples,
                                       station_idx_map[station], month_idx)
        row_probs.append(prob)
        freezing_probs.append({
            'station': station,
            'month': month_names[month_idx],
            'prob': prob,
            'ci_lo': ci[0],
            'ci_hi': ci[1]
        })
    print(f"{station:<20} {row_probs[0]:>9.0%} {row_probs[1]:>9.0%} {row_probs[2]:>9.0%} {row_probs[3]:>9.0%}")

freezing_df = pd.DataFrame(freezing_probs)

In [ ]:
# Visualize freezing probabilities as heatmap
fig, ax = plt.subplots(figsize=(10, 8))

# Pivot for heatmap
pivot = freezing_df.pivot(index='station', columns='month', values='prob')
pivot = pivot[['January', 'February', 'March', 'April']]  # Order columns

# Sort by latitude (north to south)
lat_order = ['Bergland Dam', 'Gwinn Sawyer AFB', 'Iron Mountain', 'Atlanta MI',
             'Traverse City', 'Bad Axe', 'Pontiac Airport', 'Ann Arbor UMich']
pivot = pivot.reindex(lat_order)

sns.heatmap(pivot * 100, annot=True, fmt='.0f', cmap='Blues', 
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Probability (%)'})

ax.set_title('Probability of Freezing Temperature (T < 32°F)\nOrdered North to South', fontsize=14)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Station', fontsize=12)

plt.tight_layout()
plt.savefig('plots/18_freezing_probability.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: plots/18_freezing_probability.png")

## B2. Agriculture: Safe Planting Probability

**Decision**: Farmers need to know when it's safe to plant (low frost risk).

Safe planting typically requires T > 40°F with high probability.

In [ ]:
def calc_safe_planting_prob(alpha_samples, beta_samples, sigma_samples,
                            station_idx, month_idx, threshold=40):
    """
    Calculate P(Temperature > threshold) for safe planting.
    """
    mu = alpha_samples[:, station_idx] + beta_samples[:, month_idx]
    prob = 1 - stats.norm.cdf(threshold, loc=mu, scale=sigma_samples)
    return prob.mean(), np.percentile(prob, [2.5, 97.5])

# Focus on spring months (March, April) for planting decisions
print("SAFE PLANTING PROBABILITY P(T > 40°F)")
print("="*70)
print("\nRecommendation: Plant when probability > 70%")
print(f"\n{'Station':<20} {'March':>12} {'April':>12} {'Recommendation':>20}")
print("-"*70)

planting_recs = []
for station in station_names:
    mar_prob, _ = calc_safe_planting_prob(alpha_samples, beta_samples, sigma_samples,
                                          station_idx_map[station], 2)  # March = index 2
    apr_prob, _ = calc_safe_planting_prob(alpha_samples, beta_samples, sigma_samples,
                                          station_idx_map[station], 3)  # April = index 3
    
    if apr_prob > 0.7:
        rec = "April: SAFE"
    elif mar_prob > 0.7:
        rec = "March: SAFE"
    else:
        rec = "Wait until May"
    
    planting_recs.append({
        'station': station,
        'march_prob': mar_prob,
        'april_prob': apr_prob,
        'recommendation': rec
    })
    
    mar_str = f"{mar_prob:.0%}" + (" ✓" if mar_prob > 0.7 else "")
    apr_str = f"{apr_prob:.0%}" + (" ✓" if apr_prob > 0.7 else "")
    print(f"{station:<20} {mar_str:>12} {apr_str:>12} {rec:>20}")

planting_df = pd.DataFrame(planting_recs)

In [ ]:
# Visualize planting recommendations on map
fig, ax = plt.subplots(figsize=(12, 14))

# Michigan outline
lp_lon = [-84.5, -82.5, -82.4, -83.0, -84.0, -86.5, -87.0, -86.5, -84.5]
lp_lat = [41.7, 41.7, 43.0, 44.0, 45.8, 45.8, 44.0, 43.0, 41.7]
up_lon = [-90.5, -89.0, -87.0, -84.5, -84.0, -85.5, -87.5, -90.5]
up_lat = [46.5, 46.0, 45.8, 45.8, 46.5, 47.0, 47.0, 46.5]

ax.fill(lp_lon, lp_lat, color='lightgreen', alpha=0.3, edgecolor='black', linewidth=1.5)
ax.fill(up_lon, up_lat, color='lightgreen', alpha=0.3, edgecolor='black', linewidth=1.5)

# Color by April planting probability
for _, row in planting_df.iterrows():
    lat, lon = coords[row['station']]
    if row['april_prob'] > 0.7:
        color = 'green'
        marker = '✓'
    elif row['april_prob'] > 0.5:
        color = 'orange'
        marker = '?'
    else:
        color = 'red'
        marker = '✗'
    
    ax.scatter(lon, lat, s=400, c=color, edgecolors='black', linewidths=2, zorder=5, alpha=0.8)
    ax.annotate(f"{row['station']}\nApril: {row['april_prob']:.0%}", (lon, lat),
                xytext=(10, 10), textcoords='offset points', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9))

ax.set_xlim(-91, -82)
ax.set_ylim(41.5, 47.5)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Agricultural Decision Support: Safe Planting in April\n'
             'Green = >70% probability T > 40°F', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', edgecolor='black', label='Safe (>70%)'),
    Patch(facecolor='orange', edgecolor='black', label='Marginal (50-70%)'),
    Patch(facecolor='red', edgecolor='black', label='Wait (<50%)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('plots/19_planting_decision_map.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: plots/19_planting_decision_map.png")

## B3. Road Safety: Expected Icy Days

In [ ]:
# Calculate expected number of icy days per month
# Assume ~30 days per month, each day has P(T < 32) probability of being icy

days_per_month = [31, 28, 31, 30]  # Jan, Feb, Mar, Apr

print("EXPECTED ICY DAYS PER MONTH")
print("="*70)
print(f"\n{'Station':<20} {'Jan':>8} {'Feb':>8} {'Mar':>8} {'Apr':>8} {'Total':>8}")
print("-"*70)

icy_days_data = []
for station in station_names:
    total_icy = 0
    row_icy = []
    for month_idx in range(4):
        prob, _ = calc_freezing_prob(alpha_samples, beta_samples, sigma_samples,
                                      station_idx_map[station], month_idx)
        expected_icy = prob * days_per_month[month_idx]
        row_icy.append(expected_icy)
        total_icy += expected_icy
        
        icy_days_data.append({
            'station': station,
            'month': month_names[month_idx],
            'expected_icy_days': expected_icy
        })
    
    print(f"{station:<20} {row_icy[0]:>8.1f} {row_icy[1]:>8.1f} {row_icy[2]:>8.1f} {row_icy[3]:>8.1f} {total_icy:>8.1f}")

icy_df = pd.DataFrame(icy_days_data)

In [ ]:
# Visualize icy days
fig, ax = plt.subplots(figsize=(12, 6))

pivot_icy = icy_df.pivot(index='station', columns='month', values='expected_icy_days')
pivot_icy = pivot_icy[['January', 'February', 'March', 'April']]
pivot_icy = pivot_icy.reindex(lat_order)

pivot_icy.plot(kind='bar', ax=ax, width=0.8, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Station (North to South)', fontsize=12)
ax.set_ylabel('Expected Icy Days', fontsize=12)
ax.set_title('Road Safety Planning: Expected Days Below Freezing\n'
             'Use for salt/sand budget allocation', fontsize=14)
ax.legend(title='Month', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('plots/20_icy_days_budget.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: plots/20_icy_days_budget.png")

## Summary: Decision Support Applications

In [ ]:
print("\n" + "="*70)
print("DECISION SUPPORT FRAMEWORK SUMMARY")
print("="*70)

print("""
This Bayesian hierarchical model provides uncertainty-aware decision support:

1. ENERGY PLANNING
   - Freezing probability maps help predict heating demand
   - Northern Michigan (Bergland, Gwinn): 70-95% freezing prob in Jan-Feb
   - Southern Michigan (Ann Arbor, Pontiac): 40-60% freezing prob
   - Use for: Fuel procurement, grid load forecasting

2. AGRICULTURAL PLANNING  
   - Safe planting requires P(T > 40°F) > 70%
   - Southern Michigan: April is generally safe
   - Northern Michigan: May recommended for sensitive crops
   - Use for: Planting schedules, crop insurance assessment

3. ROAD SAFETY PLANNING
   - Expected icy days inform salt/sand budgets
   - Bergland Dam: ~85 icy days (Jan-Apr)
   - Ann Arbor: ~45 icy days (Jan-Apr)
   - Use for: Winter maintenance resource allocation

KEY ADVANTAGE OF BAYESIAN APPROACH:
   - Provides full probability distributions, not just point estimates
   - Enables risk-based decision making (e.g., "90% sure it's safe")
   - Uncertainty is properly propagated through the model
""")

---
## Conclusion

This notebook demonstrates two practical applications of the Bayesian hierarchical model:

### A. Missing Data Prediction
- Hierarchical models can predict missing data by borrowing strength
- Useful for new stations or data gaps

### B. Decision Support
- Convert model outputs to actionable probabilities
- Support energy, agriculture, and transportation planning
- Bayesian approach provides uncertainty quantification for risk management